# Check the coverage of a city with the number of tasks and area
- Given the edges for correction of the Jurisdiction and the tasks geojson of the Jurisdiction, 
- Calculate how many tasks have to be reset.
- Calculate the percentage of the total area for that jurisdiction needs to be remapped.

In [40]:
import geopandas as gpd
import pandas as pd

def get_tasks_area(tasks_file:str, change_edges_file:str):
    tasks_gdf = gpd.read_file(tasks_file)
    change_edges_gdf = gpd.read_file(change_edges_file)
    
    if 'ext:addition' not in change_edges_gdf.columns or 'ext:deletion' not in change_edges_gdf.columns:
        print("Required columns 'ext:addition' or 'ext:deletion' not found in change_edges_gdf")
        print(change_edges_gdf.columns)
        return
    # Calculate the area
    tasks_gdf['area_km2'] = tasks_gdf.geometry.to_crs(tasks_gdf.estimate_utm_crs()).area / 10**6
    # ext:addition , ext:deletion
    addition_edges_gdf = change_edges_gdf[change_edges_gdf['ext:addition'] == 1]
    deleted_edges_gdf = change_edges_gdf[change_edges_gdf['ext:deletion'] == 1]
    # Get the feature ids of tasks_gdf that intersect with addition_edges_gdf and deleted_edges_gdf
    addition_task_ids = gpd.sjoin(tasks_gdf, addition_edges_gdf, how="inner", predicate='intersects') 
    deleted_task_ids = gpd.sjoin(tasks_gdf, deleted_edges_gdf, how="inner", predicate='intersects') 
    # Get the dictionary of taskId and tm_project_id for addition and deletion tasks
    deleted_task_ids['combinedId'] = deleted_task_ids['taskId'].astype(str)+ "_" + deleted_task_ids['tm_project_id']
    addition_task_ids['combinedId'] = addition_task_ids['taskId'].astype(str)+ "_" + addition_task_ids['tm_project_id']
    
    # Add both addition and deletion task ids to a single dictionary
    combined_task_ids = pd.concat([addition_task_ids, deleted_task_ids])
    # Remove duplicates based on combinedId for the combined task ids as well
    combined_task_ids = combined_task_ids.drop_duplicates(subset=['combinedId'])
    # print(combined_task_ids[['taskId','tm_project_id','geometry','area_km2']])
    # combined_task_ids.to_file('../data-refresh/combined-tasks.geojson', driver='GeoJSON')
    total_area = tasks_gdf['area_km2'].sum()
    print(f"Total area of all tasks: {total_area:.2f} km²")
    correction_area = combined_task_ids['area_km2'].sum()
    print(f"Total area of correction tasks: {correction_area:.2f} km²")
    print(f'Percentage of total area covered by correction tasks: {correction_area/total_area*100:.2f}%')
        

tasks_file = '../data-refresh/city-tasks/spokane-tasks.geojson'
change_edges_file = '../data-refresh/changed-data/spokane_city.edges.geojson'

get_tasks_area(tasks_file, change_edges_file)

Required columns 'ext:addition' or 'ext:deletion' not found in change_edges_gdf
Index(['ext:at_intersection', 'ext:corner_id', 'ext:from_intersection',
       'ext:line_description', 'ext:line_type', 'ext:node_description',
       'ext:side', 'ext:sidewalk_corrected', 'ext:sw_id',
       'ext:to_intersection', 'ext:width_confidence', 'footway', 'highway',
       'surface', 'crossing:markings', 'width', '_id', 'ext:osm_version',
       'length', 'incline', '_u_id', '_v_id', 'geometry'],
      dtype='object')
